In [ ]:
import socket
import random
import time

HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port

def tcp_client():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:
        client_socket.connect((HOST, PORT))
        print(f"[Client] Connected to {HOST}:{PORT}")
        number = 0
        while True:
            # Generate and send random data
            number += 1 #random.randint(100, 999)
            message = "Hello " + f"{number}"
            client_socket.sendall(message.encode())
            print(f"[Client] Sent: {message}")
            #time.sleep(1)
            # Receive echoed response
            for i in range(1):
                data = client_socket.recv(1024)
                print(i)
                if not data:
                    print("[Client] Connection closed by server.")
                    break
                print(f"[Client] Received: {data.decode()}")
            #time.sleep(10)  # Wait before next send

if __name__ == "__main__":
    tcp_client()


[Client] Connected to 192.168.1.10:7
[Client] Sent: Hello 1


In [5]:
import socket
import random
import time

HOST = '192.168.1.10'  # Remote device IP (server IP)
PORT = 7               # Echo port
def int_to_4byte_array(num):
    if not (0 <= num <= 0xFFFFFFFF):
        raise ValueError("Integer must be between 0 and 2^32 - 1 (inclusive)")
    return num.to_bytes(4, byteorder='big')

from collections import Counter

def print_element_counts(lst):
    counts = Counter(lst)
    for value, count in counts.items():
        print(f"Value {value} occurs {count} times")

def compare_lists(list1, list2):
    if len(list1) != len(list2):
        print("Failure: Lists are of different lengths.")
        return
    fail = 0
    for i, (a, b) in enumerate(zip(list1, list2)):
        if a != b:
            print(f"Failure: Mismatch at index {i}: {a} != {b}")
            fail = 1
            
    if(fail):
        return 1
    print("Success: All elements match.")
    return 0

def tcp_client():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as client_socket:
        client_socket.connect((HOST, PORT))
        print(f"[Client] Connected to {HOST}:{PORT}")
        #client_socket.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        total_length = 5000000
        message = bytes([64,0])+int_to_4byte_array(total_length)
        client_socket.sendall(message)

        data = client_socket.recv(1)
        data_int = int.from_bytes(data, byteorder='big');
        print(f"[Client] Received: {(data_int)}")
        packet_size = 1446
        
        payload_size = packet_size - 1
        import numpy as np

        random_array = np.random.randint(0, 256, size=total_length, dtype=np.uint8)
        #random_array = [i % 256 for i in range(total_length)]
        data_to_send = bytearray(random_array)

        for i in range(0, len(data_to_send), payload_size):
            chunk = data_to_send[i:i + payload_size]
            packet = bytearray([192]) + chunk  # First byte is 192
            client_socket.sendall(packet)
            #print(f"Sent packet {i // payload_size + 1}: {len(packet)} bytes")
            #time.sleep(0.1)
        
        data = client_socket.recv(1)
        data_int = int.from_bytes(data, byteorder='big');
        print(f"[Client] Received: {(data_int)}")

       
        data = client_socket.recv(1)
        data_int = int.from_bytes(data, byteorder='big');
        print(f"[Client] FN: {(data_int)}")

        data = client_socket.recv(4)
        data_size = int.from_bytes(data, byteorder='little');
        print(f"[Client] Received: {(data_size)}")
        recv_buf = client_socket.getsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF)
        print(f"Receive buffer size: {recv_buf} bytes")
        data_byte = bytes(0)
        i = 0
        while i != data_size:
            data = client_socket.recv(1446*2)
            data_byte = data_byte + data;
            i+=len(data)
            #print(i)
            #print(len(data))
            #time.sleep(0.01)
       #data_int = int.from_bytes(data_byte[0:1], byteorder='big');
        data_int = list(data_byte)
        #print_element_counts(data_int)
        ret = compare_lists(data_int,list(data_to_send))
        return ret
        #print(data_int)
        #print(list(random_array))

if __name__ == "__main__":
    check_list = [];
    for i in range(0,1):
        check_list.append(tcp_client())
    print_element_counts(check_list)

[Client] Connected to 192.168.1.10:7
[Client] Received: 128
[Client] Received: 128
[Client] FN: 0
[Client] Received: 5000000
Receive buffer size: 65536 bytes
Success: All elements match.
Value 0 occurs 1 times
